# Supervised Learning Classification Assignment

## Project: Breast Cancer Diagnosis Classification

This notebook uses the public **Breast Cancer Wisconsin Diagnostic Dataset** from `sklearn.datasets`. The goal is to classify tumors as **malignant** or **benign** using supervised machine learning.

The assignment introduction mentions regression, but the detailed instructions and grading rubric require classification models, classification metrics, confusion matrices, ROC curves, deployment, and monitoring. Therefore, this solution follows the classification requirements.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("Libraries imported successfully.")

## 2. Dataset Selection and Loading

The Breast Cancer Wisconsin Diagnostic Dataset is suitable because it is a binary classification dataset with real numeric medical measurements. The target variable identifies whether a tumor is malignant or benign.

In [ ]:
cancer = load_breast_cancer()

X_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df = X_df.copy()
df["target"] = cancer.target
df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})

print("Dataset loaded successfully.")
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Quick Check of Data

In [ ]:
print("First five rows:")
display(df.head())

print("Last five rows:")
display(df.tail())

print("Data types:")
display(df.dtypes)

print("Descriptive statistics:")
display(df.describe())

print("Feature names:")
print(list(cancer.feature_names))

print("Target classes:")
print(dict(enumerate(cancer.target_names)))

### Feature and Target Explanation

- The independent features are continuous numerical measurements.
- The target variable is categorical/binary: `0 = malignant`, `1 = benign`.
- This is a supervised classification problem.

## 4. Data Cleaning and Preprocessing

In [ ]:
missing_values = df.isnull().sum()
print("Missing values per column:")
display(missing_values[missing_values > 0])

duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

df_clean = df.drop_duplicates().copy()
print(f"Shape after removing duplicates: {df_clean.shape}")

X = df_clean[cancer.feature_names]
y = df_clean["target"]

print("Features and target prepared successfully.")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## 5. Exploratory Data Analysis (EDA)

### 5.1 Distribution of Classes

In [ ]:
class_counts = df_clean["diagnosis"].value_counts()
print("Class distribution:")
display(class_counts)

plt.figure(figsize=(7, 5))
plt.bar(class_counts.index, class_counts.values)
plt.title("Distribution of Diagnosis Classes")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.show()

**Insight:** The classes are not perfectly balanced, so accuracy alone is not enough. Precision, recall, F1-score, and ROC-AUC are also important.

### 5.2 Correlation Between Features

In [ ]:
correlation_matrix = df_clean.drop(columns=["diagnosis"]).corr()

plt.figure(figsize=(14, 10))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0, linewidths=0.2)
plt.title("Correlation Heatmap of Features and Target")
plt.show()

**Insight:** Several features are strongly correlated, especially measurements related to radius, perimeter, and area. This may help classification but can also create redundancy among features.

### 5.3 Feature Relationships by Diagnosis

In [ ]:
selected_features = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]

for feature in selected_features:
    plt.figure(figsize=(8, 5))
    groups = [
        df_clean[df_clean["diagnosis"] == "malignant"][feature],
        df_clean[df_clean["diagnosis"] == "benign"][feature]
    ]
    plt.boxplot(groups, labels=["malignant", "benign"])
    plt.title(f"{feature.title()} by Diagnosis")
    plt.xlabel("Diagnosis")
    plt.ylabel(feature.title())
    plt.show()

**Insight:** Malignant tumors generally show higher values for radius, perimeter, and area. These differences should help the models separate the two classes.

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Data split completed.")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

## 7. Model Implementation and Hyperparameter Tuning

Two classification models are implemented: Logistic Regression and Random Forest. GridSearchCV is used to tune key hyperparameters.

### 7.1 Logistic Regression

In [ ]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

logistic_params = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__solver": ["liblinear", "lbfgs"]
}

logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_params,
    scoring="f1",
    cv=5,
    n_jobs=1
)

logistic_grid.fit(X_train, y_train)
best_logistic_model = logistic_grid.best_estimator_

print("Best Logistic Regression parameters:")
print(logistic_grid.best_params_)

### 7.2 Random Forest

In [ ]:
random_forest = RandomForestClassifier(random_state=42)

forest_params = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

forest_grid = GridSearchCV(
    estimator=random_forest,
    param_grid=forest_params,
    scoring="f1",
    cv=5,
    n_jobs=1
)

forest_grid.fit(X_train, y_train)
best_forest_model = forest_grid.best_estimator_

print("Best Random Forest parameters:")
print(forest_grid.best_params_)

## 8. Model Evaluation

In [ ]:
def evaluate_model(model, model_name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    }

    print(f"Classification Report - {model_name}")
    print(classification_report(y_test, y_pred, target_names=cancer.target_names))
    return metrics

logistic_metrics = evaluate_model(best_logistic_model, "Logistic Regression", X_test, y_test)
forest_metrics = evaluate_model(best_forest_model, "Random Forest", X_test, y_test)

results_df = pd.DataFrame([logistic_metrics, forest_metrics])
results_df

### 8.1 Model Comparison

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
x_axis = np.arange(len(metrics_to_plot))
width = 0.35

logistic_scores = results_df[results_df["Model"] == "Logistic Regression"][metrics_to_plot].iloc[0].values
forest_scores = results_df[results_df["Model"] == "Random Forest"][metrics_to_plot].iloc[0].values

plt.figure(figsize=(10, 6))
plt.bar(x_axis - width/2, logistic_scores, width, label="Logistic Regression")
plt.bar(x_axis + width/2, forest_scores, width, label="Random Forest")
plt.title("Model Performance Comparison")
plt.xlabel("Metric")
plt.ylabel("Score")
plt.xticks(x_axis, metrics_to_plot)
plt.ylim(0, 1.05)
plt.legend()
plt.show()

### 8.2 Confusion Matrices

In [ ]:
models = {
    "Logistic Regression": best_logistic_model,
    "Random Forest": best_forest_model
}

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cancer.target_names)
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

### 8.3 ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
for model_name, model in models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=model_name, ax=plt.gca())

plt.title("ROC Curves for Classification Models")
plt.show()

## 9. Model Interpretation and Inference

In [ ]:
best_model_row = results_df.sort_values(by=["F1-score", "ROC-AUC"], ascending=False).iloc[0]
best_model_name = best_model_row["Model"]

if best_model_name == "Logistic Regression":
    best_model = best_logistic_model
else:
    best_model = best_forest_model

print("Best model based on F1-score and ROC-AUC:")
print(best_model_row)
print(f"Selected best model: {best_model_name}")

**Interpretation:** The best model is selected using F1-score and ROC-AUC because they provide stronger insight than accuracy alone. In a medical classification problem, recall is especially important because missing a malignant case could be serious. Precision is also important to reduce unnecessary false alarms.

## 10. Feature Importance

In [ ]:
if best_model_name == "Random Forest":
    importance_values = best_model.feature_importances_
    importance_df = pd.DataFrame({"Feature": X.columns, "Importance": importance_values})
else:
    coefficients = best_model.named_steps["model"].coef_[0]
    importance_df = pd.DataFrame({"Feature": X.columns, "Importance": np.abs(coefficients)})

importance_df = importance_df.sort_values(by="Importance", ascending=False).head(10)

print("Top 10 important features:")
display(importance_df)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"][::-1], importance_df["Importance"][::-1])
plt.title(f"Top 10 Important Features - {best_model_name}")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 11. Prediction on New Data

In [ ]:
new_sample = pd.DataFrame([X.median()], columns=X.columns)

new_prediction = best_model.predict(new_sample)[0]
new_probability = best_model.predict_proba(new_sample)[0]
prediction_label = cancer.target_names[new_prediction]

print("New sample prediction:")
print(f"Predicted class: {prediction_label}")
print(f"Probability of malignant: {new_probability[0]:.4f}")
print(f"Probability of benign: {new_probability[1]:.4f}")

## 12. Deployment and Monitoring

### Deployment Plan
The best-performing model can be deployed as a REST API using Flask or FastAPI. The model and preprocessing pipeline can be saved using `joblib`. A front-end application or internal system can send patient feature values to the API and receive a predicted class and probability.

### Real-Time Data Handling
Incoming data should be validated to ensure all required features are present, numeric, and within reasonable ranges. Invalid or missing inputs should be rejected or flagged before prediction.

### Potential Issues
- Data drift if future patient data differs from training data.
- Performance decay if real-world patterns change.
- Bias if the training data does not represent all patient populations.
- Latency if predictions are needed in real time.

### Monitoring Strategy
Track input data distributions, predicted class proportions, prediction confidence, API latency, and model performance when true labels become available. Retrain the model periodically with updated validated data.

## 13. Conclusion

This project completed a full supervised classification workflow: dataset selection, preprocessing, EDA, model training, hyperparameter tuning, evaluation, interpretation, prediction, and deployment planning. Logistic Regression and Random Forest were compared using multiple metrics, and the best model was selected based on F1-score and ROC-AUC.